# 第2回 宿題（発展）: 講座の用語集で LoRA 学習し、用語集の検索（RAG）と比べる

演習ノートブック（`lora_format_tuning.ipynb`）と **同じモデル・同じ LoRA の設定** で、架空 FAQ の代わりに講座の用語集から作った Q&A（`scripts/make_sft_dataset.py`、58 語 × 2 通りの質問 = 116 件）を学習します。
そのうえで、自分で用意した評価用の質問 34 問に、次の条件で答えさせて比べます。

| 条件 | モデル | 入力 |
| --- | --- | --- |
| `base` | 学習前 | 質問だけ |
| `rag` | 学習前 | 用語集から検索した項目（最大 3 件）＋質問 |
| `ft` | 学習後（LoRA） | 質問だけ（学習データと同じ形） |
| `ft_rag`（任意） | 学習後（LoRA） | 用語集から検索した項目＋質問 |

評価用の質問（`lectures/02/glossary_eval.json`、`scripts/make_glossary_eval.py` で作成）は 4 区分です。

| 区分 | 件数 | 何を見るか | 正解の判定 |
| --- | --- | --- | --- |
| `seen` | 12 | 学習データと同じ言い回し。覚えたか | 定義の要点の語句を含む |
| `paraphrase` | 12 | 学習データに無い言い回し（別名で聞く `alias` / 定義から用語名を聞く `reverse`） | 同上 |
| `unknown_course` | 5 | 講座に固有らしいが用語集に無い架空の用語。「分からない」と言えるか | 「記載がありません」「分かりません」などを含む |
| `general` | 5 | 用語集に無い一般的な用語（RAG、Transformer など）。元の能力が壊れていないか | 要点の語句を含む |

`seen` と `paraphrase` の 12 語は、講座に固有の用語 6 語（宿題の3レベル、lecNN-start タグ、既定モデル、answer.md、Pi、TestModel）と、一般的な用語 6 語（LoRA、temperature など）です。

**判定は語句の一致による機械的なもの** です。言い換えで正しく答えても × になったり、要点の語句だけ含んで中身が誤っていても ○ になったりします。最後の表で出力の原文を読んで確かめてください。

**時間の目安（T4）**: セクション 0〜3 で約 10 分（評価 2 条件を含む）、学習は約 8〜10 分（116 件 × 5 エポック = 75 ステップ。演習の 40 ステップ・4〜5 分からの見積もり）、学習後の評価で約 3 分。実際の時間は各セルの表示で確かめてください。

セルは上から順に実行します。設定を変えたら **ランタイム → セッションを再起動** して上から実行し直してください。

## 0. 環境の確認とライブラリの導入

演習ノートブックと同じです。1 つ目のセルに `Tesla T4` と表示され、2 つ目のセルの最後に版の一覧が出れば準備完了です。pip の赤い `ERROR: pip's dependency resolver ...` は無視してかまいません。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# 演習ノートブックと同じ版（Colab の T4 で動作確認済みの版）に固定する
PINS = {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao は peft 0.21 と非互換なので外す
%pip install -q {spec}

import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

演習ノートブックから変えたのは次の 4 点だけです。モデル・revision・学習率・LoRA の設定は同じです。

- `num_train_epochs`: 20 → **5**。学習データが 14 件から 116 件に増えたので、学習時間を T4 で 10 分前後に収めるため。1 エポック 15 ステップ、5 エポックで 75 ステップです（各例を 5 回学習）。
- `max_new_tokens`: 96 → **160**。用語の説明は FAQ の JSON より長いため。
- `rag_top_k` / `rag_threshold`: 用語集の検索で渡す件数と、渡す最低スコア。
- `data_base_url`: データを取ってくる場所（自分の fork の `lec02` ブランチ）。同じ名前のファイルを Colab にアップロードしてあれば、そちらを使います。

`RUN_FT_RAG = True` にすると、学習後のモデルにも検索結果を渡す条件（`ft_rag`）を追加で評価します（約 3 分増えます）。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # 演習と同じ重み
    "seed": 42,
    "num_train_epochs": 5,
    "max_steps": -1,            # -1 ならエポック数で決める（動作確認で短く回すときだけ正の数にする）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 160,
    "gen": {"do_sample": False},   # 貪欲生成で、条件の違い以外の揺れを除く
    "rag_top_k": 3,
    "rag_threshold": 0.25,
    "data_base_url": "https://raw.githubusercontent.com/T-Kodai/lecture-ai-engineering-2026/lec02/",
}
RUN_FT_RAG = False
WORK = Path("/content/glossary_lab") if Path("/content").exists() else Path("glossary_lab")
WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. データの読み込み

3 つのファイルを読み込みます。Colab にアップロードしたファイル（左のファイル一覧の `/content`）があればそれを、無ければ `data_base_url` から取ってきます。

| ファイル | 中身 |
| --- | --- |
| `lectures/02/glossary_sft.jsonl` | 学習データ。`uv run python scripts/make_sft_dataset.py > lectures/02/glossary_sft.jsonl` の出力（116 件、`messages` 形式） |
| `lectures/02/glossary_eval.json` | 評価用の質問 34 問と正解の語句 |
| `data/glossary.json` | 用語集そのもの。検索（RAG）の条件で使う |

**見るところ**: `train=116 eval=34` と区分ごとの件数。`seen` の質問はすべて学習データにあり、それ以外は 1 問も学習データに無いことを確かめます（違えばエラーになります）。

In [ ]:
import urllib.request

def load_text(rel_path: str) -> str:
    """ローカル（アップロード済み）にあればそれを、無ければ data_base_url から読む"""
    name = Path(rel_path).name
    for p in [Path(rel_path), Path("/content") / name, Path(name)]:
        if p.exists():
            print(f"読み込み: {p}")
            return p.read_text(encoding="utf-8")
    url = CONFIG["data_base_url"] + rel_path
    print(f"ダウンロード: {url}")
    with urllib.request.urlopen(url) as r:
        return r.read().decode("utf-8")

train_rows = [json.loads(line) for line in load_text("lectures/02/glossary_sft.jsonl").splitlines() if line.strip()]
EVAL = json.loads(load_text("lectures/02/glossary_eval.json"))
QUESTIONS, REFUSAL_MARKERS = EVAL["questions"], EVAL["refusal_markers"]
TERMS = json.loads(load_text("data/glossary.json"))["terms"]

train_questions = {r["messages"][0]["content"] for r in train_rows}
for q in QUESTIONS:
    in_train = q["question"] in train_questions
    assert in_train == (q["category"] == "seen"), f"学習データとの重なりが想定と違います: {q['question']}"

from collections import Counter
print(f"train={len(train_rows)} eval={len(QUESTIONS)} terms={len(TERMS)}", dict(Counter(q["category"] for q in QUESTIONS)))
print("学習例の 1 件:", train_rows[0]["messages"])

## 3. モデルの読み込み、検索、採点、学習前の評価

- モデルは演習と同じく量子化なし（32bit）で読み込みます。
- **検索（RAG の条件）**: 質問に用語名・別名がそのまま含まれていれば強く一致とし、それに加えて質問と定義の 2 文字の重なり（多くの定義に出る 2 文字ほど軽く数える）で点数を付け、上位 `rag_top_k` 件のうち `rag_threshold` 以上を渡します。ベクトル検索ではない、単純な文字の一致です。
- **RAG の指示**: 「次の用語集の項目だけを根拠に答え、無ければ『用語集に記載がありません』と答える」をシステムプロンプトに書きます。`base` と `ft` は学習データと同じく質問だけを渡します。
- **採点**: `unknown_course` は「分からない」を表す語句を含めば ○、それ以外は正解の語句グループをすべて含めば ○。別に、どの条件でも「分からない」と答えたか（`refused`）を記録します。

**見るところ**: `base` と `rag` の要約（区分ごとの正解率）と、検索が正しい用語を拾えた割合（`retrieval_hit`）。評価は 1 条件あたり 34 問で数分かかります。

In [ ]:
import math, re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", round(model.get_memory_footprint() / 1e9, 2))

def generate(model, messages):
    # enable_thinking=False: 演習と同じく、思考ブロックを空にして回答だけを生成させる
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# ---- 用語集の検索（RAG の条件で使う）----
def norm(s):
    return re.sub(r"[\s「」『』（）()、。？?！!・:：]", "", s.lower())

def names(t):
    base = re.split(r"[（(]", t["term"])[0]
    inner = re.findall(r"[（(]([^）)]+)[）)]", t["term"])
    return {n for n in [t["term"], base, *inner, *t["aliases"]] if len(norm(n)) >= 2}

# 質問によく付く言い回し。検索の手がかりにならないので除く
QUESTION_STOP = ["とは何ですか", "について教えてください", "について簡単に説明してください", "を何と呼びますか",
                 "は何ですか", "はどんな", "ですか", "ください", "aie2026の", "aie2026"]

def bigrams(s, strip_question=False):
    s = norm(s)
    if strip_question:
        for w in QUESTION_STOP: s = s.replace(norm(w), "")
    return {s[i:i + 2] for i in range(len(s) - 1)}

DEF_BIGRAMS = {t["id"]: bigrams(t["term"] + t["definition"]) for t in TERMS}
DF = Counter(b for bs in DEF_BIGRAMS.values() for b in bs)
def idf(b): return math.log((len(TERMS) + 1) / (DF.get(b, 0) + 1))   # 多くの定義に出る 2 文字ほど軽くする

def name_hit(name, q):
    n, qn = norm(name), norm(q)
    if re.fullmatch(r"[a-z0-9._\-]+", n):   # 英数字の名前は、前後が英数字でないときだけ一致（api の中の pi などを避ける）
        return re.search(rf"(?<![a-z0-9]){re.escape(n)}(?![a-z0-9])", qn) is not None
    return n in qn

def retrieve(q):
    """用語名・別名の一致と、定義との 2 文字の重なり（IDF で重み付け）で用語を選ぶ"""
    qb = bigrams(q, strip_question=True)
    total = sum(idf(b) for b in qb) or 1
    scored = []
    for t in TERMS:
        overlap = sum(idf(b) for b in qb & DEF_BIGRAMS[t["id"]]) / total
        hits = [n for n in names(t) if name_hit(n, q)]
        bonus = 0.5 + max(len(norm(n)) for n in hits) / 100 if hits else 0
        scored.append((overlap + bonus, t))
    scored.sort(key=lambda x: -x[0])
    return [t for s, t in scored[:CONFIG["rag_top_k"]] if s >= CONFIG["rag_threshold"]]

RAG_SYSTEM = ("あなたは AIE2026 講座の用語集の回答係です。次の用語集の項目だけを根拠に、日本語で簡潔に答えてください。"
              "用語集に答えが無いときは「用語集に記載がありません」と答えてください。\n\n用語集:\n{entries}")

def build_messages(question, condition):
    if condition.endswith("rag"):
        hits = retrieve(question)
        entries = "\n".join(f"- {t['term']}: {t['definition']}" for t in hits) or "（該当する項目はありません）"
        return [{"role": "system", "content": RAG_SYSTEM.format(entries=entries)}, {"role": "user", "content": question}], [t["id"] for t in hits]
    return [{"role": "user", "content": question}], []   # 学習データと同じ形（システムプロンプトなし）

# ---- 採点 ----
def has_any(text, group):
    return any(w.lower() in text.lower() for w in group)

def judge(q, raw):
    refused = any(m in raw for m in REFUSAL_MARKERS)
    ok = refused if q["category"] == "unknown_course" else all(has_any(raw, g) for g in q["must_include"])
    return ok, refused

CATEGORIES = ["seen", "paraphrase", "unknown_course", "general"]

def evaluate(model, condition):
    rows, t0 = [], time.time()
    for q in QUESTIONS:
        messages, retrieved = build_messages(q["question"], condition)
        raw = generate(model, messages)
        ok, refused = judge(q, raw)
        rows.append({**q, "condition": condition, "raw": raw, "ok": ok, "refused": refused, "retrieved": retrieved,
                     "retrieval_hit": (q["term_id"] in retrieved) if (condition.endswith("rag") and q["term_id"]) else None})
    summary = {"condition": condition, "n": len(rows), "seconds": round(time.time() - t0, 1),
               "ok_rate": round(sum(r["ok"] for r in rows) / len(rows), 3)}
    for c in CATEGORIES:
        rs = [r for r in rows if r["category"] == c]
        summary[f"{c}_ok"] = round(sum(r["ok"] for r in rs) / len(rs), 3)
        summary[f"{c}_refused"] = round(sum(r["refused"] for r in rs) / len(rs), 3)
    hits = [r["retrieval_hit"] for r in rows if r["retrieval_hit"] is not None]
    if hits: summary["retrieval_hit"] = round(sum(hits) / len(hits), 3)
    (WORK / f"eval_{condition}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

results = {}
for cond in ["base", "rag"]:
    results[cond] = evaluate(model, cond)

## 4. LoRA 学習

演習と同じ `LoraConfig` と `SFTConfig`（バッチ 4 × 勾配累積 2、cosine、warmup 6）で、116 件を 5 エポック学習します。学習データは `messages` 形式なので、`Dataset.from_list` にそのまま渡せます。

**見るところ**: `trainable%`、`Training Loss`（10 ステップごと）、最後の行の学習時間とステップ数（75 ステップ）。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_rows)
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], max_steps=CONFIG["max_steps"],
                per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=25, save_total_limit=2,
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
train_out = trainer.train()   # 途中で止まったら trainer.train(resume_from_checkpoint=True) で再開できる
TRAIN_SECONDS = round(time.time() - t0)
LOSSES = [{"step": h["step"], "loss": h["loss"]} for h in trainer.state.log_history if "loss" in h]
print(f"学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f} | 平均 loss: {train_out.training_loss:.3f} | loss の推移: {LOSSES}")
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 5. 学習後の評価

学習前と同じ 34 問・同じ生成設定で、学習後のモデルに質問だけを渡して評価します（`ft`）。`RUN_FT_RAG = True` なら、検索結果も渡す条件（`ft_rag`）も評価します。

In [ ]:
results["ft"] = evaluate(tuned, "ft")
if RUN_FT_RAG:
    results["ft_rag"] = evaluate(tuned, "ft_rag")

## 6. 比較と提出

- **区分ごとの正解率**: 条件ごとに並べます。`refused` は「分からない」と答えた割合です（`general` で高ければ、答えられる質問まで断っていることになります）。
- **退行**: 学習前（`base`）は ○ だったのに、学習後（`ft`）に × になった質問。
- **質問ごとの表**: 出力の原文です。語句の判定が妥当か、自分の目で確かめてください。

`result.json` と `eval_*.json` が `glossary_lab/`（Colab では `/content/glossary_lab`）にできます。左のファイル一覧で右クリック → ダウンロードし、`lectures/02/results/` に置いて PR に含めます。

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

conds = list(results)
summary_table = pd.DataFrame([results[c][0] for c in conds]).set_index("condition")
display(summary_table)

rows_by = {c: {r["id"]: r for r in results[c][1]} for c in conds}
regressed = [q["question"] for q in QUESTIONS if rows_by["base"][q["id"]]["ok"] and not rows_by["ft"][q["id"]]["ok"]]
improved = [q["question"] for q in QUESTIONS if not rows_by["base"][q["id"]]["ok"] and rows_by["ft"][q["id"]]["ok"]]
print("退行（base ○ → ft ×）:", regressed)
print("改善（base × → ft ○）:", improved)

def mark(ok): return "○" if ok else "×"
detail = pd.DataFrame([{"区分": q["category"], "種類": q["kind"], "質問": q["question"],
                        **{f"{c}": f"{mark(rows_by[c][q['id']]['ok'])} {rows_by[c][q['id']]['raw']}" for c in conds},
                        "検索した用語": ", ".join(rows_by["rag"][q["id"]]["retrieved"])} for q in QUESTIONS])
display(detail)

result = {
    "config": CONFIG,
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "device": DEVICE,
    "method": "LoRA（量子化なし）",
    "data": {"train": len(train_rows), "eval": len(QUESTIONS), "eval_by_category": dict(Counter(q["category"] for q in QUESTIONS))},
    "train": {"seconds": TRAIN_SECONDS, "steps": trainer.state.global_step, "epochs": trainer.state.epoch, "mean_loss": train_out.training_loss, "losses": LOSSES},
    "summaries": {c: results[c][0] for c in conds},
    "regressed": regressed,
    "improved": improved,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存しました:", sorted(p.name for p in WORK.glob("*.json")))